# Generación de Figuras — Paper Multi-Agente LLM
**Autor:** Edurne Martínez de Contrasta | PROJENER.AI SL · UAX 2026

Este notebook genera las **18 figuras** del paper en formato PDF de alta resolución (300 dpi).
Guarda todos los PDFs en la misma carpeta donde esté este notebook.
Después copia esa carpeta junto al `.tex` y recompila con MiKTeX.

---
### Instrucciones de uso
1. Coloca este notebook en `C:\Users\edurn\practicas projener\` (misma carpeta que el `.tex`)
2. Ejecuta **Kernel → Restart & Run All**
3. Verifica que aparecen los 18 archivos `.pdf` en la misma carpeta
4. Recompila el `.tex` con MiKTeX

In [1]:
# ── INSTALACIÓN Y CONFIGURACIÓN ───────────────────────────────────────────────
# Ejecutar solo si falta algún paquete
# !pip install matplotlib numpy scipy

import matplotlib
matplotlib.use('Agg')   # backend sin pantalla — compatible con cualquier entorno
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# ── ESTILO GLOBAL (aspecto revista Q1) ────────────────────────────────────────
plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'font.size':          10,
    'axes.titlesize':     11,
    'axes.labelsize':     10,
    'xtick.labelsize':    9,
    'ytick.labelsize':    9,
    'legend.fontsize':    9,
    'figure.dpi':         150,
    'savefig.dpi':        300,
    'savefig.bbox':       'tight',
    'savefig.pad_inches': 0.05,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          True,
    'grid.alpha':         0.3,
    'grid.linestyle':     '--',
})

# ── PALETA DE COLORES ─────────────────────────────────────────────────────────
# Colores asignados a cada variante (consistentes en todas las figuras)
COLORS = {
    'RPA-Baseline':       '#555555',
    'Single-Small':       '#2196F3',   # azul
    'Single-Small-XL':    '#03A9F4',   # azul claro
    'Multi-Small':        '#FF9800',   # naranja
    'Multi-Small-Abl':    '#FFC107',   # ámbar
    'Multi-Mixed':        '#4CAF50',   # verde  ← ÓPTIMO
    'Multi-Large':        '#8BC34A',   # verde claro
    'Multi-Mixed-HiL':    '#009688',   # teal
    'Multi-Mixed-HiL-Clf':'#673AB7',   # morado
}

# ── DATOS CENTRALES DEL PAPER ─────────────────────────────────────────────────
modelos = [
    'RPA-Baseline', 'Single-Small', 'Single-Small-XL',
    'Multi-Small', 'Multi-Small-Abl', 'Multi-Mixed',
    'Multi-Large', 'Multi-Mixed-HiL', 'Multi-Mixed-HiL-Clf'
]
modelos_cortos = [
    'RPA-Base', 'Single-S', 'Single-XL',
    'Multi-S', 'Multi-Abl', 'Multi-Mix',
    'Multi-L', 'Mix-HiL', 'Mix-HiL-Clf'
]

ARR = [100, 92, 79.0, 79.5, 88, 98.7, 98, 98, 94]
HDA = [  0, 10.3, 36.5, 17.3, 0, 5.1, 0, 0, 23.1]
DER = [ 34, 27.3, 20.5, 27.5, 30, 27.3, 26, 28, 22]
PT  = [  0, 0.31, 0.40, 1.74, 0.77, 1.69, 2.40, 1.56, 2.85]
TOK = [  0,  200,  750,  750,  450,  750,  750,  875,  200]
COST= [0.000, 0.0012, 0.0045, 0.0045, 0.0027, 0.0065, 0.0120, 0.0053, 0.0024]

colores_barras = [COLORS[m] for m in modelos]

OBJETIVO_ARR = 70
OBJETIVO_HDA = 90
OBJETIVO_DER = 10

print('✅ Configuración cargada. Generando figuras...')

✅ Configuración cargada. Generando figuras...


---
## Figura 1 — Comparativa de métricas principales (ARR, HDA, DER)

In [21]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Only models RPA-Baseline → Multi-Mixed-HiL (first 8)
m8 = modelos_cortos[:8]
c8 = colores_barras[:8]
x = np.arange(len(m8))
w = 0.65

datos = [
    (ARR[:8], 'ARR — Autonomous Resolution Rate (%)', OBJETIVO_ARR, '>70%'),
    (HDA[:8], 'HDA — HiL Detection Accuracy (%)',     OBJETIVO_HDA, '>90%'),
    (DER[:8], 'DER — Decision Error Rate (%)',         OBJETIVO_DER, '<10%'),
]

for ax, (vals, titulo, obj, lbl_obj) in zip(axes, datos):
    bars = ax.bar(x, vals, width=w, color=c8, edgecolor='white', linewidth=0.5)
    ax.axhline(obj, color='red', linestyle='--', linewidth=1.2,
               label=f'Target {lbl_obj}')
    ax.set_title(titulo, fontsize=10, pad=6)
    ax.set_xticks(x)
    ax.set_xticklabels(m8, rotation=40, ha='right', fontsize=8)
    ax.set_ylim(0, 115)
    ax.set_ylabel('%')
    ax.legend(fontsize=8)
    # labels above bars
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f'{v:.0f}', ha='center', va='bottom', fontsize=7.5)

plt.suptitle('Main metrics comparison by system variant',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('figura1.pdf')
plt.show()
print('✅ figura1.pdf saved')

✅ figura1.pdf saved


---
## Figura 2 — Curva de eficiencia ARR vs. Tiempo de proceso (PT)

In [22]:
fig, ax = plt.subplots(figsize=(8, 5))

for i, (m, mc) in enumerate(zip(modelos, modelos_cortos)):
    color = COLORS[m]
    marker = 'D' if m == 'Multi-Mixed' else ('s' if m == 'Single-Small' else 'o')
    size = 120 if m in ('Multi-Mixed', 'Single-Small') else 70
    ax.scatter(PT[i], ARR[i], color=color, s=size, marker=marker,
               zorder=5, edgecolors='black', linewidths=0.5)
    offset_x = 0.04
    offset_y = 1.2
    if m == 'RPA-Baseline':  offset_x = 0.04; offset_y = -3.5
    if m == 'Multi-Mixed':   offset_x = 0.04; offset_y =  1.5
    ax.annotate(mc, (PT[i], ARR[i]),
                xytext=(PT[i]+offset_x, ARR[i]+offset_y),
                fontsize=8, color=color)

ax.axhline(OBJETIVO_ARR, color='red', linestyle='--', linewidth=1,
           label='Target ARR > 70%')
ax.set_xlabel('Processing time PT (s/case)')
ax.set_ylabel('ARR — Autonomous Resolution Rate (%)')
ax.set_title('Efficiency frontier: ARR vs. processing time per variant')
ax.set_xlim(-0.15, 3.2)
ax.set_ylim(55, 108)
ax.legend()

# Optimal zone annotation
ax.annotate('Optimal zone\n(high ARR, low PT)',
            xy=(0.35, 92), xytext=(0.6, 72),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('figura2.pdf')
plt.show()
print('✅ figura2.pdf saved')

✅ figura2.pdf saved


---
## Figura 3 — Distribución de decisiones por modelo

In [6]:
# Distribución estimada de decisiones sobre 50 casos
# Aprobado / Rechazado / Escalado-HiL / Error
# ARR = (Aprobado+Rechazado)/50*100, HDA aplica sobre 13 casos HiL
dist_data = {
    'RPA-Baseline':       [33, 17,  0,  0],   # ARR=100%, HDA=0%, DER=34% → 17 err implícito
    'Single-Small':       [26, 12,  4,  8],
    'Single-Small-XL':    [28, 14,  3,  5],
    'Multi-Small':        [20, 14,  4, 12],
    'Multi-Small-Abl':    [28, 16,  0,  6],
    'Multi-Mixed':        [31, 18,  1,  0],
    'Multi-Large':        [30, 19,  0,  1],
    'Multi-Mixed-HiL':    [30, 19,  0,  1],
    'Multi-Mixed-HiL-Clf':[29, 18,  3,  0],
}
categorias = ['Aprobado', 'Rechazado', 'Escalado HiL', 'Error']
cat_colors  = ['#4CAF50', '#F44336', '#FF9800', '#9E9E9E']

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(modelos))
w = 0.18
offsets = [-1.5*w, -0.5*w, 0.5*w, 1.5*w]

for j, (cat, col, off) in enumerate(zip(categorias, cat_colors, offsets)):
    vals = [dist_data[m][j] for m in modelos]
    ax.bar(x + off, vals, width=w, label=cat, color=col,
           edgecolor='white', linewidth=0.4)

ax.set_xticks(x)
ax.set_xticklabels(modelos_cortos, rotation=35, ha='right')
ax.set_ylabel('Número de casos (sobre 50)')
ax.set_title('Distribución de decisiones por variante del sistema (50 casos)')
ax.legend(ncol=4, loc='upper right')
ax.set_ylim(0, 42)

plt.tight_layout()
plt.savefig('figura3.pdf')
plt.show()
print('✅ figura3.pdf guardada')

✅ figura3.pdf guardada


---
## Figura 4 — Gráfico radar de perfiles de rendimiento

In [24]:
from matplotlib.patches import FancyArrowPatch

# Dimensions: ARR, HDA, Precision (100-DER), Speed (inverse PT normalized)
# Normalized 0-100 for comparable radar
def normalizar_velocidad(pt_vals):
    max_pt = max(v for v in pt_vals if v > 0)
    return [100 * (1 - v/max_pt) if v > 0 else 100 for v in pt_vals]

vel = normalizar_velocidad(PT)
prec = [100 - d for d in DER]

# Only 5 representative models for readability
modelos_radar = ['RPA-Baseline', 'Single-Small', 'Multi-Small', 'Multi-Mixed', 'Multi-Mixed-HiL-Clf']
idx_radar = [modelos.index(m) for m in modelos_radar]

dimensiones = ['ARR\n(>70%)', 'HDA\n(>90%)', 'Precision\n(>90%)', 'Speed\n(max)']
N = len(dimensiones)
angulos = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angulos += angulos[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for i, m in zip(idx_radar, modelos_radar):
    valores = [ARR[i], HDA[i], prec[i], vel[i]]
    valores += valores[:1]
    ax.plot(angulos, valores, 'o-', linewidth=1.8,
            color=COLORS[m], label=m)
    ax.fill(angulos, valores, alpha=0.07, color=COLORS[m])

# Minimum target line
obj_vals = [70, 90, 90, 50]
obj_vals += obj_vals[:1]
ax.plot(angulos, obj_vals, '--', color='red', linewidth=1, alpha=0.6, label='Minimum target')

ax.set_xticks(angulos[:-1])
ax.set_xticklabels(dimensiones, fontsize=9)
ax.set_ylim(0, 105)
ax.set_yticks([25, 50, 75, 100])
ax.set_yticklabels(['25', '50', '75', '100'], fontsize=7)
ax.set_title('Multi-dimensional performance profiles', pad=18, fontsize=11)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8)

plt.tight_layout()
plt.savefig('figura4.pdf')
plt.show()
print('✅ figura4.pdf saved')

✅ figura4.pdf saved


---
## Figura 5 — ARR vs. Coste en tokens + Consumo de tokens por modelo

In [23]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: ARR vs cost
for i, (m, mc) in enumerate(zip(modelos, modelos_cortos)):
    ax1.scatter(COST[i], ARR[i], color=COLORS[m], s=90,
                zorder=5, edgecolors='black', linewidths=0.5)
    ax1.annotate(mc, (COST[i], ARR[i]),
                 xytext=(COST[i]+0.0001, ARR[i]+1),
                 fontsize=7.5, color=COLORS[m])

ax1.axhline(OBJETIVO_ARR, color='red', linestyle='--', linewidth=1,
            label='Target ARR > 70%')
ax1.set_xlabel('Estimated cost per case ($)')
ax1.set_ylabel('ARR (%)')
ax1.set_title('ARR vs. computational cost')
ax1.legend(fontsize=8)

# Right: tokens per model
x = np.arange(len(modelos))
bars = ax2.bar(x, TOK, color=colores_barras, edgecolor='white', linewidth=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(modelos_cortos, rotation=35, ha='right')
ax2.set_ylabel('Tokens per case')
ax2.set_title('Estimated token consumption per variant')

for bar, v in zip(bars, TOK):
    if v > 0:
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+8,
                 str(v), ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('figura5.pdf')
plt.show()
print('✅ figura5.pdf saved')

✅ figura5.pdf saved


---
## Figura 6 — Topología DAG + Centralidad de agentes

In [7]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ── Izquierda: DAG lineal con betweenness ──────────────────────────────────────
agentes = ['Requester', 'Procurement', 'Finance', 'Legal', 'Compliance']
betweenness = [0, 3, 4, 3, 0]
posiciones_x = [0, 1.8, 3.6, 5.4, 7.2]
pos_y = 0.5

node_colors = ['#B0BEC5', '#FF9800', '#F44336', '#FF9800', '#4CAF50']
node_sizes = [1200 + b*400 for b in betweenness]

ax1.set_xlim(-0.8, 8.0)
ax1.set_ylim(0, 1)
ax1.axis('off')

# Flechas
for i in range(len(agentes)-1):
    ax1.annotate('', xy=(posiciones_x[i+1]-0.35, pos_y),
                 xytext=(posiciones_x[i]+0.35, pos_y),
                 arrowprops=dict(arrowstyle='->', color='#555', lw=2))

# Nodos
for i, (ag, px, nc, ns, bw) in enumerate(
        zip(agentes, posiciones_x, node_colors, node_sizes, betweenness)):
    circle = plt.Circle((px, pos_y), 0.25, color=nc, zorder=5,
                         linewidth=2, edgecolor='white')
    ax1.add_patch(circle)
    ax1.text(px, pos_y, f'BC={bw}', ha='center', va='center',
             fontsize=8, fontweight='bold', color='white', zorder=6)
    ax1.text(px, pos_y - 0.33, ag, ha='center', va='top',
             fontsize=8.5, fontweight='bold')

ax1.set_title('Topología de comunicación (DAG lineal)\nBetweenness Centrality por nodo',
              fontsize=10)

# ── Derecha: gráfico de barras de centralidad ──────────────────────────────────
x = np.arange(len(agentes))
grado_total   = [1, 2, 2, 2, 1]
grado_entrada = [0, 1, 1, 1, 1]

w = 0.28
b1 = ax2.bar(x - w, grado_total,   width=w, label='Grado total',
             color='#2196F3', edgecolor='white')
b2 = ax2.bar(x,     grado_entrada, width=w, label='Grado entrada',
             color='#FF9800', edgecolor='white')
b3 = ax2.bar(x + w, betweenness,   width=w, label='Betweenness',
             color='#F44336', edgecolor='white')

ax2.set_xticks(x)
ax2.set_xticklabels(agentes, rotation=20, ha='right')
ax2.set_ylabel('Valor de la métrica')
ax2.set_title('Métricas de centralidad en la red de comunicación')
ax2.legend()
ax2.set_ylim(0, 5.5)

plt.tight_layout()
plt.savefig('figura6.pdf')
plt.show()
print('✅ figura6.pdf guardada')

✅ figura6.pdf guardada


---
## Figura 7 — Tasa de resolución por nivel de complejidad

In [8]:
modelos_comp = ['Single-Small', 'Multi-Small', 'Multi-Mixed', 'Multi-Mixed-HiL', 'Multi-Small-Abl']
simple  = [100,  95, 100, 100,  95]
medio   = [100,83.3, 100,94.4,88.9]
complejo= [100,83.3, 100, 100,83.3]

x = np.arange(len(modelos_comp))
w = 0.26

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w, simple,   width=w, label='Simple (n=20)',
            color='#4CAF50', edgecolor='white')
b2 = ax.bar(x,     medio,    width=w, label='Medio (n=18)',
            color='#FF9800', edgecolor='white')
b3 = ax.bar(x + w, complejo, width=w, label='Complejo (n=12)',
            color='#F44336', edgecolor='white')

for bars in (b1, b2, b3):
    for bar in bars:
        v = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, v + 0.5,
                f'{v:.0f}', ha='center', va='bottom', fontsize=7.5)

ax.axhline(OBJETIVO_ARR, color='red', linestyle='--', linewidth=1,
           label='Objetivo > 70%')
ax.set_xticks(x)
ax.set_xticklabels(modelos_comp, rotation=15, ha='right')
ax.set_ylabel('Tasa de resolución correcta (%)')
ax.set_title('Tasa de resolución por nivel de complejidad del caso')
ax.set_ylim(70, 108)
ax.legend()

plt.tight_layout()
plt.savefig('figura7.pdf')
plt.show()
print('✅ figura7.pdf guardada')

✅ figura7.pdf guardada


---
## Figura 8 — Multi-Mixed-HiL vs. Multi-Mixed-HiL-Clf

In [9]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# ── Izquierda: comparativa de métricas ────────────────────────────────────────
metricas = ['ARR', 'HDA', 'DER']
hil_vals  = [98,   0,  28]
clf_vals  = [94,  23.1, 22]
x = np.arange(len(metricas))
w = 0.35

b1 = ax1.bar(x - w/2, hil_vals, width=w, label='Multi-Mixed-HiL\n(reglas de prompt)',
             color=COLORS['Multi-Mixed-HiL'], edgecolor='white')
b2 = ax1.bar(x + w/2, clf_vals, width=w, label='Multi-Mixed-HiL-Clf\n(clasificador incertidumbre)',
             color=COLORS['Multi-Mixed-HiL-Clf'], edgecolor='white')

for bars in (b1, b2):
    for bar in bars:
        v = bar.get_height()
        ax1.text(bar.get_x()+bar.get_width()/2, v + 0.8,
                 f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

ax1.set_xticks(x)
ax1.set_xticklabels(metricas)
ax1.set_ylabel('%')
ax1.set_ylim(0, 115)
ax1.set_title('Multi-Mixed-HiL vs. Multi-Mixed-HiL-Clf\nImpacto del clasificador de incertidumbre')
ax1.legend(fontsize=8)

# Anotaciones de delta
deltas = [clf_vals[i] - hil_vals[i] for i in range(3)]
delta_labels = [f'{d:+.1f} pp' for d in deltas]
delta_colors = ['#F44336' if d < 0 else '#4CAF50' for d in deltas]
for i, (dl, dc) in enumerate(zip(delta_labels, delta_colors)):
    ax1.text(i, max(hil_vals[i], clf_vals[i]) + 7, dl,
             ha='center', fontsize=9, color=dc, fontweight='bold')

# ── Derecha: casos HiL detectados ─────────────────────────────────────────────
categorias_pie = ['Detectados\ncorrectamente', 'No detectados']
tamanios = [3, 10]  # de 13 casos HiL en total
cols_pie = ['#4CAF50', '#FFCCBC']
explode = (0.06, 0)

wedges, texts, autotexts = ax2.pie(
    tamanios, labels=categorias_pie, autopct='%1.0f%%',
    colors=cols_pie, explode=explode, startangle=90,
    textprops={'fontsize': 9})
autotexts[0].set_fontweight('bold')
ax2.set_title('Multi-Mixed-HiL-Clf: casos HiL\n(13 casos que requieren escalación)', fontsize=10)
ax2.text(0, -1.45, '0 falsos positivos  •  HDA = 23.1%',
         ha='center', fontsize=8.5, style='italic', color='#555')

plt.tight_layout()
plt.savefig('figura8.pdf')
plt.show()
print('✅ figura8.pdf guardada')

✅ figura8.pdf guardada


---
## Figura 9 — Robustez estadística: barras de error (3 réplicas)

In [10]:
modelos_rep = ['RPA-Baseline', 'Single-Small', 'Multi-Mixed']
col_rep = [COLORS[m] for m in modelos_rep]

medias_ARR = [100.0, 92.0, 98.7]
sigma_ARR  = [  0.0, 11.3,  0.9]
medias_HDA = [  0.0, 10.3,  5.1]
sigma_HDA  = [  0.0, 14.5,  3.6]
medias_DER = [ 34.0, 27.3, 27.3]
sigma_DER  = [  0.0,  3.8,  0.9]

x = np.arange(len(modelos_rep))
w = 0.55

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
datasets = [
    (medias_ARR, sigma_ARR, 'ARR — Tasa Resolución Autónoma (%)', OBJETIVO_ARR, '>70%'),
    (medias_HDA, sigma_HDA, 'HDA — Precisión Detección HiL (%)',  OBJETIVO_HDA, '>90%'),
    (medias_DER, sigma_DER, 'DER — Tasa Error Decisión (%)',       OBJETIVO_DER, '<10%'),
]

for ax, (medias, sigmas, titulo, obj, lbl_obj) in zip(axes, datasets):
    bars = ax.bar(x, medias, width=w, color=col_rep,
                  edgecolor='white', linewidth=0.5, alpha=0.85)
    ax.errorbar(x, medias, yerr=sigmas, fmt='none',
                color='black', capsize=6, capthick=1.5, linewidth=1.5, zorder=6)
    ax.axhline(obj, color='red', linestyle='--', linewidth=1,
               label=f'Objetivo {lbl_obj}')
    ax.set_title(titulo, fontsize=9.5)
    ax.set_xticks(x)
    ax.set_xticklabels(modelos_rep, rotation=20, ha='right', fontsize=8.5)
    ax.set_ylabel('%')
    ax.set_ylim(0, 120)
    ax.legend(fontsize=8)
    # Etiqueta media ± sigma
    for i, (m, s) in enumerate(zip(medias, sigmas)):
        label = f'{m:.1f}\n±{s:.1f}pp' if s > 0 else f'{m:.1f}\n(σ=0)'
        ax.text(i, m + sigmas[i] + 3, label,
                ha='center', fontsize=7.5, color=col_rep[i])

plt.suptitle('Robustez estadística: media ± desviación estándar sobre 3 réplicas independientes',
             fontsize=10, y=1.01)
plt.tight_layout()
plt.savefig('figura9.pdf')
plt.show()
print('✅ figura9.pdf guardada')

✅ figura9.pdf guardada


---
## Figura 10 (figura_inc1_metrics) — Experimento 02: Gestión de Incidencias

In [25]:
def figura_experimento(fname, titulo, arr_vals, hda_vals, der_vals, modelos_exp, note=None):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
    x = np.arange(len(modelos_exp))
    w = 0.55
    col = [COLORS[m] for m in modelos_exp]
    datos = [
        (arr_vals, 'ARR — Autonomous Resolution Rate (%)', OBJETIVO_ARR, '>70%'),
        (hda_vals, 'HDA — HiL Detection Accuracy (%)',     OBJETIVO_HDA, '>90%'),
        (der_vals, 'DER — Decision Error Rate (%)',         OBJETIVO_DER, '<10%'),
    ]
    for ax, (vals, subtit, obj, lbl_obj) in zip(axes, datos):
        bars = ax.bar(x, vals, width=w, color=col, edgecolor='white', linewidth=0.5)
        ax.axhline(obj, color='red', linestyle='--', linewidth=1, label=f'Target {lbl_obj}')
        ax.set_xticks(x)
        ax.set_xticklabels(modelos_exp, rotation=20, ha='right', fontsize=9)
        ax.set_title(subtit, fontsize=9.5)
        ax.set_ylabel('%')
        ax.set_ylim(0, 115)
        ax.legend(fontsize=8)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
                    f'{v:.0f}', ha='center', va='bottom', fontsize=9)
    plt.suptitle(titulo, fontsize=11, y=1.01)
    if note:
        fig.text(0.5, -0.03, note, ha='center', fontsize=8, style='italic', color='#555')
    plt.tight_layout()
    plt.savefig(f'{fname}.pdf')
    plt.show()
    print(f'✅ {fname}.pdf saved')

# Experiment 02 — Incident Management
figura_experimento(
    'figura_inc1_metrics',
    'Experiment 02 — Incident Management (20 cases)',
    arr_vals=[100, 40, 45],
    hda_vals=[0, 100, 100],
    der_vals=[15,   0,   0],
    modelos_exp=['RPA-Baseline', 'Single-Small', 'Multi-Mixed'],
    note='HDA=100% with Single-Small and Multi-Mixed — only experiment reaching the HDA>90% target'
)

✅ figura_inc1_metrics.pdf saved


---
## Figura 11 (figura_inc2_crossprocess) — Cross-process Procurement vs. Incidencias

In [12]:
procesos = ['Procurement\n(Exp.01)', 'Incidencias\n(Exp.02)']
arr_proc = {'Single-Small': [92, 40],  'Multi-Mixed': [98.7, 45]}
hda_proc = {'Single-Small': [10.3, 100], 'Multi-Mixed': [5.1, 100]}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
x = np.arange(len(procesos))
w = 0.35

for ax, data, titulo, obj, lbl_obj in [
    (ax1, arr_proc, 'ARR por proceso (%)', OBJETIVO_ARR, '>70%'),
    (ax2, hda_proc, 'HDA por proceso (%)', OBJETIVO_HDA, '>90%')
]:
    for j, (m, vals) in enumerate(data.items()):
        offset = (j - 0.5) * w
        bars = ax.bar(x + offset, vals, width=w, label=m,
                      color=COLORS[m], edgecolor='white')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, v+1.5,
                    f'{v:.0f}', ha='center', fontsize=9)
    ax.axhline(obj, color='red', linestyle='--', linewidth=1, label=f'Obj. {lbl_obj}')
    ax.set_xticks(x)
    ax.set_xticklabels(procesos)
    ax.set_ylabel('%')
    ax.set_ylim(0, 115)
    ax.set_title(titulo)
    ax.legend(fontsize=8)

# Anotación flecha HDA
ax2.annotate('', xy=(1, 100), xytext=(0, 10.3),
             arrowprops=dict(arrowstyle='->', color='#673AB7', lw=2))
ax2.text(0.5, 60, '+89.7 pp HDA\n(señales binarias\nvs. regulatorias)',
         ha='center', fontsize=8, color='#673AB7', style='italic')

plt.suptitle('Comparativa cross-process: Procurement vs. Gestión de Incidencias', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('figura_inc2_crossprocess.pdf')
plt.show()
print('✅ figura_inc2_crossprocess.pdf guardada')

✅ figura_inc2_crossprocess.pdf guardada


---
## Figuras 12-13 — Experimento 03: Onboarding

In [13]:
# Exp 03 métricas
figura_experimento(
    'figura_onb1_metrics',
    'Experimento 03 — Onboarding de Clientes (20 casos)',
    arr_vals=[100, 75, 70],
    hda_vals=[0, 83.3, 100],
    der_vals=[70, 45, 40],
    modelos_exp=['RPA-Baseline', 'Single-Small', 'Multi-Mixed'],
    note='Señales mixtas (PEP, AML, UBO): HDA intermedia entre procurement (10.3%) e incidencias (100%)'
)

✅ figura_onb1_metrics.pdf guardada


In [14]:
# Gradiente 3 procesos
fig, ax = plt.subplots(figsize=(9, 5))

procesos3 = ['Procurement\n(Regulatoria\nimplícita)', 'Onboarding\n(Mixta\nPEP/AML)', 'Incidencias\n(Binaria\nexplícita)']
hda_ss3 = [10.3, 83.3, 100]
hda_mm3 = [5.1, 100, 100]
x3 = np.arange(3)

ax.plot(x3, hda_ss3, 'o-', color=COLORS['Single-Small'], linewidth=2.5,
        markersize=10, label='Single-Small')
ax.plot(x3, hda_mm3, 's-', color=COLORS['Multi-Mixed'], linewidth=2.5,
        markersize=10, label='Multi-Mixed')

for i, (v1, v2) in enumerate(zip(hda_ss3, hda_mm3)):
    ax.annotate(f'{v1:.1f}%', (i, v1), xytext=(i+0.05, v1+3),
                fontsize=9, color=COLORS['Single-Small'])
    ax.annotate(f'{v2:.1f}%', (i, v2), xytext=(i+0.05, v2-7),
                fontsize=9, color=COLORS['Multi-Mixed'])

ax.axhline(OBJETIVO_HDA, color='red', linestyle='--', linewidth=1, label='Objetivo HDA > 90%')
ax.set_xticks(x3)
ax.set_xticklabels(procesos3)
ax.set_ylabel('HDA — Precisión Detección HiL (%)')
ax.set_title('Gradiente de explicitud de señales: HDA por tipo de dominio\n(3 primeros experimentos)')
ax.set_ylim(0, 115)
ax.legend()

# Anotación gradiente
ax.annotate('', xy=(2, 95), xytext=(0, 15),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5,
                            connectionstyle='arc3,rad=-0.2'))
ax.text(1, 30, 'Señales más explícitas\n→ HDA aumenta', ha='center',
        fontsize=8.5, color='gray', style='italic')

plt.tight_layout()
plt.savefig('figura_onb2_hda_gradient.pdf')
plt.show()
print('✅ figura_onb2_hda_gradient.pdf guardada')

✅ figura_onb2_hda_gradient.pdf guardada


In [15]:
# Cross-process 3 experimentos
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

proc3 = ['Procurement', 'Onboarding', 'Incidencias']
arr3 = {'RPA-Baseline':[100,100,100], 'Single-Small':[92,75,40], 'Multi-Mixed':[98.7,70,45]}
hda3 = {'RPA-Baseline':[0,0,0], 'Single-Small':[10.3,83.3,100], 'Multi-Mixed':[5.1,100,100]}

x3 = np.arange(3)
w3 = 0.26
offsets3 = [-w3, 0, w3]

for ax, data, titulo, obj, lbl_obj in [
    (ax1, arr3, 'ARR por proceso (%)', OBJETIVO_ARR, '>70%'),
    (ax2, hda3, 'HDA por proceso (%)', OBJETIVO_HDA, '>90%')
]:
    for (m, vals), off in zip(data.items(), offsets3):
        bars = ax.bar(x3+off, vals, width=w3, label=m, color=COLORS[m], edgecolor='white')
    ax.axhline(obj, color='red', linestyle='--', linewidth=1, label=f'Obj. {lbl_obj}')
    ax.set_xticks(x3)
    ax.set_xticklabels(proc3)
    ax.set_ylabel('%')
    ax.set_ylim(0, 115)
    ax.set_title(titulo)
    ax.legend(fontsize=8)

plt.suptitle('Comparativa cross-process ARR y HDA — 3 experimentos', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('figura_onb3_crossprocess.pdf')
plt.show()
print('✅ figura_onb3_crossprocess.pdf guardada')

✅ figura_onb3_crossprocess.pdf guardada


---
## Figuras 14-15 — Experimento 04: Compliance Check

In [16]:
figura_experimento(
    'figura_comp1_metrics',
    'Experimento 04 — Compliance Check de Proveedores (20 casos)',
    arr_vals=[100, 70, 65],
    hda_vals=[0, 83.3, 100],
    der_vals=[70, 40, 35],
    modelos_exp=['RPA-Baseline', 'Single-Small', 'Multi-Mixed'],
    note='Señales mixtas (OFAC, GDPR, accidente laboral): patrón consistente con onboarding'
)

✅ figura_comp1_metrics.pdf guardada


In [17]:
# Gradiente 4 procesos
fig, ax = plt.subplots(figsize=(10, 5))

proc4 = ['Procurement\n(IR)', 'Compliance\n(MX)', 'Onboarding\n(MX)', 'Incidencias\n(EB)']
hda_ss4 = [10.3, 83.3, 83.3, 100]
hda_mm4 = [5.1, 100, 100, 100]
x4 = np.arange(4)

ax.plot(x4, hda_ss4, 'o-', color=COLORS['Single-Small'], linewidth=2.5, markersize=10, label='Single-Small')
ax.plot(x4, hda_mm4, 's-', color=COLORS['Multi-Mixed'],  linewidth=2.5, markersize=10, label='Multi-Mixed')

for i, (v1, v2) in enumerate(zip(hda_ss4, hda_mm4)):
    ax.annotate(f'{v1:.1f}%', (i, v1), xytext=(i+0.05, v1+3), fontsize=9, color=COLORS['Single-Small'])
    if v2 != v1:
        ax.annotate(f'{v2:.1f}%', (i, v2), xytext=(i+0.05, v2-7), fontsize=9, color=COLORS['Multi-Mixed'])

ax.axhline(OBJETIVO_HDA, color='red', linestyle='--', linewidth=1, label='Objetivo HDA > 90%')

# Fondo de categorías
for i, (cat, col) in enumerate(zip(['IR', 'MX', 'MX', 'EB'],
                                    ['#FFECB3','#E8F5E9','#E8F5E9','#E3F2FD'])):
    ax.axvspan(i-0.45, i+0.45, alpha=0.25, color=col, zorder=0)

ax.set_xticks(x4)
ax.set_xticklabels(proc4)
ax.set_ylabel('HDA (%)')
ax.set_title('Gradiente de explicitud de señales de riesgo — 4 experimentos\nIR=Implícita Regulatoria · MX=Mixta · EB=Explícita Binaria')
ax.set_ylim(0, 115)
ax.legend()

plt.tight_layout()
plt.savefig('figura_comp2_hda_gradient4.pdf')
plt.show()
print('✅ figura_comp2_hda_gradient4.pdf guardada')

✅ figura_comp2_hda_gradient4.pdf guardada


---
## Figuras 16-17 — Experimento 05: Análisis Financiero

In [18]:
figura_experimento(
    'figura_fin1_metrics',
    'Experimento 05 — Análisis Financiero (20 casos)',
    arr_vals=[100, 100, 70],
    hda_vals=[0, 0, 100],
    der_vals=[60, 60, 30],
    modelos_exp=['RPA-Baseline', 'Single-Small', 'Multi-Mixed'],
    note='Señales semánticas estratégicas (SS): único proceso donde Single-Small HDA=0% — igual que RPA-Baseline'
)

✅ figura_fin1_metrics.pdf guardada


In [19]:
# HDA completa 5 procesos
fig, ax = plt.subplots(figsize=(11, 5.5))

proc5 = ['Procurement\n(IR)', 'Compliance\n(MX)', 'Onboarding\n(MX)', 'Incidencias\n(EB)', 'Financiero\n(SS)']
hda_ss5 = [10.3, 83.3, 83.3, 100, 0]
hda_mm5 = [5.1, 100, 100, 100, 100]
x5 = np.arange(5)

ax.plot(x5, hda_ss5, 'o-', color=COLORS['Single-Small'], linewidth=2.5,
        markersize=11, label='Single-Small', zorder=5)
ax.plot(x5, hda_mm5, 's-', color=COLORS['Multi-Mixed'], linewidth=2.5,
        markersize=11, label='Multi-Mixed', zorder=5)

for i, (v1, v2) in enumerate(zip(hda_ss5, hda_mm5)):
    ax.annotate(f'{v1:.1f}%', (i, v1), xytext=(i+0.07, v1+3),
                fontsize=9, color=COLORS['Single-Small'], fontweight='bold')
    if v2 != v1:
        ax.annotate(f'{v2:.1f}%', (i, v2), xytext=(i+0.07, v2-8),
                    fontsize=9, color=COLORS['Multi-Mixed'], fontweight='bold')

ax.axhline(OBJETIVO_HDA, color='red', linestyle='--', linewidth=1.2,
           label='Objetivo HDA > 90%')

# Fondo categorías señales
cat_bgcolors = ['#FFECB3', '#E8F5E9', '#E8F5E9', '#E3F2FD', '#F3E5F5']
cat_labels   = ['IR', 'MX', 'MX', 'EB', 'SS']
for i, (col, lbl) in enumerate(zip(cat_bgcolors, cat_labels)):
    ax.axvspan(i-0.45, i+0.45, alpha=0.3, color=col, zorder=0)
    ax.text(i, 108, lbl, ha='center', fontsize=8.5, fontweight='bold',
            color='#555', bbox=dict(boxstyle='round,pad=0.2', facecolor=col, edgecolor='none'))

ax.set_xticks(x5)
ax.set_xticklabels(proc5)
ax.set_ylabel('HDA — Precisión Detección HiL (%)')
ax.set_title('Gradiente completo de HDA en los 5 procesos\n'
             'IR=Impl. Regulatoria · MX=Mixta · EB=Explícita Binaria · SS=Semántica Estratégica')
ax.set_ylim(-5, 115)
ax.legend()

# Anotación financiero
ax.annotate('Single-Small = RPA-Baseline\n(HDA=0% en señales SS)',
            xy=(4, 0), xytext=(3.1, 25),
            arrowprops=dict(arrowstyle='->', color='#673AB7'),
            fontsize=8.5, color='#673AB7', style='italic')

plt.tight_layout()
plt.savefig('figura_fin2_hda_complete.pdf')
plt.show()
print('✅ figura_fin2_hda_complete.pdf guardada')

✅ figura_fin2_hda_complete.pdf guardada


---
## Verificación final — resumen de archivos generados

In [20]:
import os

figuras_esperadas = [
    'figura1.pdf', 'figura2.pdf', 'figura3.pdf', 'figura4.pdf', 'figura5.pdf',
    'figura6.pdf', 'figura7.pdf', 'figura8.pdf', 'figura9.pdf',
    'figura_inc1_metrics.pdf', 'figura_inc2_crossprocess.pdf',
    'figura_onb1_metrics.pdf', 'figura_onb2_hda_gradient.pdf', 'figura_onb3_crossprocess.pdf',
    'figura_comp1_metrics.pdf', 'figura_comp2_hda_gradient4.pdf',
    'figura_fin1_metrics.pdf', 'figura_fin2_hda_complete.pdf'
]

print('\n═══════════════════════════════════════════════════')
print('       VERIFICACIÓN DE FIGURAS GENERADAS')
print('═══════════════════════════════════════════════════')
total_ok = 0
for f in figuras_esperadas:
    existe = os.path.exists(f)
    size   = os.path.getsize(f) if existe else 0
    estado = f'✅ {size//1024} KB' if existe else '❌ FALTA'
    print(f'  {f:<45} {estado}')
    if existe: total_ok += 1

print('═══════════════════════════════════════════════════')
print(f'  Total: {total_ok}/{len(figuras_esperadas)} figuras generadas')
if total_ok == len(figuras_esperadas):
    print()
    print('  ✅ TODO OK — copia estos PDFs junto al .tex y recompila')
    print('  Comando MiKTeX:')
    print('  pdflatex projener_multiagente_ES_v3.tex')
    print('  bibtex projener_multiagente_ES_v3')
    print('  pdflatex projener_multiagente_ES_v3.tex')
    print('  pdflatex projener_multiagente_ES_v3.tex')
else:
    print()
    print('  ⚠️  Algunas figuras no se generaron. Revisa los errores arriba.')
print('═══════════════════════════════════════════════════')


═══════════════════════════════════════════════════
       VERIFICACIÓN DE FIGURAS GENERADAS
═══════════════════════════════════════════════════
  figura1.pdf                                   ✅ 21 KB
  figura2.pdf                                   ✅ 21 KB
  figura3.pdf                                   ✅ 17 KB
  figura4.pdf                                   ✅ 22 KB
  figura5.pdf                                   ✅ 21 KB
  figura6.pdf                                   ✅ 26 KB
  figura7.pdf                                   ✅ 17 KB
  figura8.pdf                                   ✅ 31 KB
  figura9.pdf                                   ✅ 22 KB
  figura_inc1_metrics.pdf                       ✅ 32 KB
  figura_inc2_crossprocess.pdf                  ✅ 28 KB
  figura_onb1_metrics.pdf                       ✅ 33 KB
  figura_onb2_hda_gradient.pdf                  ✅ 29 KB
  figura_onb3_crossprocess.pdf                  ✅ 18 KB
  figura_comp1_metrics.pdf                      ✅ 32 KB
  figura_comp2

In [23]:
import os
print(os.getcwd())

C:\Users\edurn\practicas projener
